## UrbanRide_03 — Silver Customers
**Method:** Batch transformation — Bronze → Silver  
**Input:** `urbanride.bronze.customers`  
**Output:** `urbanride.silver.customers`  
**Schedule:** Daily · runs after UrbanRide_01 (Bronze CSV ingest)

**Load strategy:**
- First run → full load — processes all Bronze rows, creates Silver table
- Daily runs → incremental — processes only rows where `_ingested_at > last _processed_at`
- Watermark: `_ingested_at` (Bronze) vs `_processed_at` (Silver)
- Target grain: one clean row per `customer_id`

**Transformations applied:**
- Deduplication on `customer_id`
- City name standardisation — dirty variants → canonical
- Invalid rating nullification + city average fill
- Churn consistency validation
- Partition by `city`

In [0]:
from pyspark.sql.functions import (
    col, when, trim, lower, current_timestamp,
    avg, round, lit, coalesce
)
from pyspark.sql.types import DoubleType

CATALOG = 'urbanride'
BRONZE  = f'{CATALOG}.bronze'
SILVER  = f'{CATALOG}.silver'

spark.sql(f'USE CATALOG {CATALOG}')

# CITY STANDARDISATION MAP 
CITY_MAP = {
    'bangalore':  'Bengaluru',
    'blr':        'Bengaluru',
    'bengaluru':  'Bengaluru',
    'hyd':        'Hyderabad',
    'hyderabad':  'Hyderabad',
    'hyderbad':   'Hyderabad',
    'delhi':      'Delhi',
    'new delhi':  'Delhi',
    'mumbai':     'Mumbai',
    'bombay':     'Mumbai',
    'pune':       'Pune',
}

def standardise_city(col_name):
    """Maps dirty city name variants to canonical form."""
    expr = when(lit(False), lit(None))
    for dirty, clean in CITY_MAP.items():
        expr = expr.when(lower(trim(col(col_name))) == dirty, clean)
    return expr.otherwise(col(col_name))

VALID_CITIES = ['Delhi', 'Mumbai', 'Bengaluru', 'Hyderabad', 'Pune']

print(f'Catalog : {CATALOG}')
print(f'Source  : {BRONZE}.customers')
print(f'Target  : {SILVER}.customers')
print(f'City map: {len(CITY_MAP)} dirty variants mapped')
print('Config ready.')


Catalog : urbanride
Source  : urbanride.bronze.customers
Target  : urbanride.silver.customers
City map: 11 dirty variants mapped
Config ready.


In [0]:
# Check if Silver table already exists
# This determines whether we do a full load or incremental append
print('Checking load type...')

table_exists = spark.catalog.tableExists(f'{SILVER}.customers')

if not table_exists:
    LOAD_TYPE   = 'full'
    last_run    = None
    print('  Silver table does not exist — FULL LOAD')
else:
    LOAD_TYPE = 'incremental'
    last_run  = spark.sql(
        f'SELECT MAX(_processed_at) FROM {SILVER}.customers'
    ).collect()[0][0]
    print(f'  Silver table exists — INCREMENTAL LOAD')
    print(f'  Last processed at : {last_run}')

print(f'  Load type : {LOAD_TYPE}')


Checking load type...
  Silver table does not exist — FULL LOAD
  Load type : full


In [0]:
print('Reading bronze.customers...')

df_bronze_full = spark.table(f'{BRONZE}.customers')

if LOAD_TYPE == 'full':
    # First run — process all Bronze rows
    df_bronze = df_bronze_full
    print(f'  Full load — reading all rows')
else:
    # Incremental — only rows ingested after last Silver run
    df_bronze = df_bronze_full.filter(col('_ingested_at') > last_run)
    print(f'  Incremental load — reading rows ingested after {last_run}')

input_count = df_bronze.count()
print(f'  Rows to process : {input_count:,}')

if input_count == 0:
    print('  No new rows to process — Silver already up to date.')
    dbutils.notebook.exit('No new rows — skipping')

# Profile before cleaning
print()
print('Bronze profile (before cleaning):')
print(f'  Null ratings     : {df_bronze.filter(col("rating").isNull()).count():,}')
print(f'  Invalid ratings  : {df_bronze.filter(col("rating") > 5.0).count():,}')
print(f'  Duplicate rows   : {input_count - df_bronze.dropDuplicates(["customer_id","signup_date"]).count():,}')
print(f'  Churned          : {df_bronze.filter(col("is_churned") == True).count():,}')


Reading bronze.customers...
  Full load — reading all rows
  Rows to process : 89,963

Bronze profile (before cleaning):
  Null ratings     : 449
  Invalid ratings  : 470
  Duplicate rows   : 922
  Churned          : 10,878


In [0]:
# Deduplicate on customer_id + signup_date
# A customer can only sign up once on a given date
print('Deduplicating...')

df_deduped = df_bronze.dropDuplicates(['customer_id', 'signup_date'])

removed = input_count - df_deduped.count()
print(f'  Rows before : {input_count:,}')
print(f'  Rows after  : {df_deduped.count():,}')
print(f'  Removed     : {removed:,} duplicate rows')


Deduplicating...
  Rows before : 89,963
  Rows after  : 89,041
  Removed     : 922 duplicate rows


In [0]:
print('Standardising city names...')

# Show dirty distribution only on full load — too verbose for incremental
if LOAD_TYPE == 'full':
    print('  City distribution before standardisation:')
    df_deduped.groupBy('city').count().orderBy('count', ascending=False).show(15, truncate=False)

df_city = df_deduped.withColumn('city', standardise_city('city'))

# Flag unknown cities — should always be 0 after standardisation
unknown = df_city.filter(~col('city').isin(VALID_CITIES))
unknown_count = unknown.count()

if LOAD_TYPE == 'full':
    print('  City distribution after standardisation:')
    df_city.groupBy('city').count().orderBy('count', ascending=False).show()

print(f'  Unknown cities remaining : {unknown_count}')
if unknown_count > 0:
    print('  WARNING — unmapped cities found:')
    unknown.select('customer_id','city').distinct().show()


Standardising city names...
  City distribution before standardisation:
+---------+-----+
|city     |count|
+---------+-----+
|Delhi    |23109|
|Mumbai   |19302|
|Bengaluru|15102|
|Hyderabad|11421|
|Pune     |7940 |
|delhi    |1762 |
|New Delhi|1748 |
|mumbai   |1508 |
|Bombay   |1503 |
|pune     |938  |
|BLR      |918  |
|bengaluru|891  |
|Bangalore|886  |
|Hyderbad |685  |
|Hyd      |664  |
+---------+-----+
only showing top 15 rows
  City distribution after standardisation:
+---------+-----+
|     city|count|
+---------+-----+
|    Delhi|26619|
|   Mumbai|22313|
|Bengaluru|17797|
|Hyderabad|13434|
|     Pune| 8878|
+---------+-----+

  Unknown cities remaining : 0


In [0]:
#imports
from pyspark.sql.functions import broadcast

# Step 1 — flag and nullify invalid ratings > 5.0
print('Cleaning ratings...')

df_rating = df_city \
    .withColumn('is_rating_invalid', col('rating') > 5.0) \
    .withColumn('rating',
        when(col('rating') > 5.0, None)
        .otherwise(col('rating'))
    )

print(f'  Invalid ratings nullified : {df_rating.filter(col("is_rating_invalid") == True).count():,}')

# Step 2 — compute city average from SILVER if incremental
# On incremental runs, use existing Silver data for city averages
# to stay consistent with previously written rows
if LOAD_TYPE == 'full':
    avg_source = df_rating
    print('  Computing city averages from incoming Bronze data (full load)')
else:
    avg_source = spark.table(f'{SILVER}.customers')
    print('  Computing city averages from existing Silver data (incremental)')

city_avg = avg_source \
    .filter(col('rating').isNotNull()) \
    .groupBy('city') \
    .agg(round(avg('rating'), 2).alias('avg_rating'))

print('  City average ratings:')
city_avg.show()

# Step 3 — fill NULL ratings with city average
df_rating_filled = df_rating \
    .join(broadcast(city_avg), on='city', how='left') \
    .withColumn('rating', coalesce(col('rating'), col('avg_rating'))) \
    .drop('avg_rating')

print(f'  Null ratings remaining : {df_rating_filled.filter(col("rating").isNull()).count()}')


Cleaning ratings...
  Invalid ratings nullified : 467
  Computing city averages from incoming Bronze data (full load)
  City average ratings:
+---------+----------+
|     city|avg_rating|
+---------+----------+
|    Delhi|      4.25|
|Bengaluru|      4.25|
|     Pune|      4.25|
|   Mumbai|      4.24|
|Hyderabad|      4.25|
+---------+----------+

  Null ratings remaining : 0


In [0]:
# Flag rows where churn logic is inconsistent
# is_churned=True  but churn_date=NULL  → inconsistent
# is_churned=False but churn_date set   → inconsistent
print('Validating churn consistency...')

df_churn = df_rating_filled.withColumn('is_churn_inconsistent',
    when(
        (col('is_churned') == True)  & col('churn_date').isNull(), True
    ).when(
        (col('is_churned') == False) & col('churn_date').isNotNull(), True
    ).otherwise(False)
)

inconsistent_count = df_churn.filter(col('is_churn_inconsistent') == True).count()
print(f'  Inconsistent churn rows : {inconsistent_count:,}')

if inconsistent_count > 0:
    df_churn.filter(col('is_churn_inconsistent') == True) \
        .select('customer_id','is_churned','churn_date') \
        .show(5, truncate=False)


Validating churn consistency...
  Inconsistent churn rows : 0


In [0]:
df_silver = df_churn \
    .withColumn('_processed_at', current_timestamp()) \
    .withColumn('_source', lit(f'{BRONZE}.customers'))

print(f'Final row count   : {df_silver.count():,}')
print(f'Final column count: {len(df_silver.columns)}')


Final row count   : 89,041
Final column count: 15


In [0]:
print({LOAD_TYPE})

{'full'}


In [0]:
import time
print(f'Writing silver.customers — mode: {LOAD_TYPE}...')
t0 = time.time()

if LOAD_TYPE == 'full':
    # First run — create Silver table with full data
    df_silver.write \
        .format('delta') \
        .mode('overwrite') \
        .partitionBy('city') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(f'{SILVER}.customers')
    print(f'  Full load written : {df_silver.count():,} rows')
else:
    # Incremental — append only new cleaned rows
    df_silver.write \
        .format('delta') \
        .mode('append') \
        .saveAsTable(f'{SILVER}.customers')
    print(f'  Incremental rows appended : {df_silver.count():,}')

print(f'  Write time : {time.time()-t0 }s')
print()

# OPTIMIZE — compact files after write
print('Running OPTIMIZE...')
spark.sql(f'OPTIMIZE {SILVER}.customers')
print('  OPTIMIZE complete')


Writing silver.customers — mode: full...
  Full load written : 89,041 rows
  Write time : 4.1761462688446045s

Running OPTIMIZE...
  OPTIMIZE complete


In [0]:
print('=== SILVER CUSTOMERS VERIFICATION ===')
print()

df_verify = spark.table(f'{SILVER}.customers')
total = df_verify.count()

print(f'  Total rows : {total:,}')
print(f'  Load type  : {LOAD_TYPE}')
print()

# Data quality checks
print('Data quality checks:')
checks = [
    ('Null customer_id',    df_verify.filter(col('customer_id').isNull()).count()),
    ('Invalid rating > 5',  df_verify.filter(col('rating') > 5.0).count()),
    ('Null rating',         df_verify.filter(col('rating').isNull()).count()),
    ('Unknown city',        df_verify.filter(~col('city').isin(VALID_CITIES)).count()),
    ('Churn inconsistency', df_verify.filter(col('is_churn_inconsistent') == True).count()),
    ('Null signup_date',    df_verify.filter(col('signup_date').isNull()).count()),
]

all_passed = True
for check, result in checks:
    status = 'PASS' if result == 0 else 'WARN'
    if result > 0: all_passed = False
    print(f'  {status}  {check:<30} : {result:,}')

print()
print('City distribution:')
df_verify.groupBy('city').count().orderBy('count', ascending=False).show()

print('Churn summary:')
df_verify.groupBy('is_churned').count().show()

print('Sample rows:')
df_verify.select(
    'customer_id','name','city','rating',
    'is_churned','churn_date','is_rating_invalid',
    'is_churn_inconsistent','_processed_at'
).limit(5).show(truncate=False)

print()
if all_passed:
    print('All quality checks passed. Silver customers ready.')
else:
    print('Some checks flagged — review WARN items above.')
print('Next: UrbanRide_04 — Silver Trips')


=== SILVER CUSTOMERS VERIFICATION ===

  Total rows : 89,041
  Load type  : full

Data quality checks:
  PASS  Null customer_id               : 0
  PASS  Invalid rating > 5             : 0
  PASS  Null rating                    : 0
  PASS  Unknown city                   : 0
  PASS  Churn inconsistency            : 0
  PASS  Null signup_date               : 0

City distribution:
+---------+-----+
|     city|count|
+---------+-----+
|    Delhi|26619|
|   Mumbai|22313|
|Bengaluru|17797|
|Hyderabad|13434|
|     Pune| 8878|
+---------+-----+

Churn summary:
+----------+-----+
|is_churned|count|
+----------+-----+
|      true|10770|
|     false|78271|
+----------+-----+

Sample rows:
+------------------------------------+-------------+-----+------+----------+----------+-----------------+---------------------+--------------------------+
|customer_id                         |name         |city |rating|is_churned|churn_date|is_rating_invalid|is_churn_inconsistent|_processed_at             |
+--

In [0]:
# display(spark.sql("DESCRIBE HISTORY urbanride.silver.customers"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-03-09T14:05:48.000Z,8815326091183894,bvishaladf@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [""city""], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3178775399414853),7d43062a-204d-416b-a523-3132d27e876a,0309-134843-ru5utkh4-v2n,0,WriteSerializable,false,"Map(numFiles -> 5, numRemovedFiles -> 5, numRemovedBytes -> 2967504, numDeletionVectorsRemoved -> 0, numOutputRows -> 89041, numOutputBytes -> 2967504)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-09T13:58:59.000Z,8815326091183894,bvishaladf@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [""city""], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3178775399414853),d46ceccd-dbe3-4eb9-81d4-7df2536735e1,0309-134843-ru5utkh4-v2n,null,WriteSerializable,false,"Map(numFiles -> 5, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 89041, numOutputBytes -> 2967504)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


In [0]:
# Current file count in the Delta table
# display(spark.sql("DESCRIBE DETAIL urbanride.silver.customers"))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,a3dcaeab-ffa3-4465-80ed-83e4dbd2e292,urbanride.silver.customers,null,,2026-03-09T13:58:53.107Z,2026-03-09T14:05:48.000Z,List(city),List(),5,2967504,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
